In [0]:
%sql
SELECT
    'Stocks' AS product,
    SUM(stocks_revenue_90d) AS revenue
FROM default.groww_day90_outcomes

UNION ALL

SELECT
    'F&O',
    SUM(fno_revenue_90d)
FROM default.groww_day90_outcomes

UNION ALL

SELECT
    'Mutual Funds',
    SUM(mf_revenue_90d)
FROM default.groww_day90_outcomes

UNION ALL

SELECT
    'IPO',
    SUM(ipo_successful_90d) * 1.0
FROM default.groww_day90_outcomes

ORDER BY revenue DESC;

In [0]:
%sql
SELECT
    high_value_90d,
    COUNT(*) AS users,
    ROUND(SUM(observable_revenue_90d), 2) AS total_revenue,
    ROUND(AVG(observable_revenue_90d), 2) AS revenue_per_user,
    ROUND(
        100.0 * SUM(observable_revenue_90d) /
        SUM(SUM(observable_revenue_90d)) OVER (),
        2
    ) AS revenue_share_pct
FROM default.groww_day90_outcomes
GROUP BY high_value_90d;

In [0]:
%sql
WITH revenue AS (
    SELECT
        SUM(stocks_revenue_90d) AS stocks_revenue,
        SUM(fno_revenue_90d) AS fno_revenue,
        SUM(mf_revenue_90d) AS mf_revenue,
        SUM(ipo_successful_90d) * 1.0 AS ipo_revenue
    FROM default.groww_day90_outcomes
),

base AS (
    SELECT
        stocks_revenue + fno_revenue +
        mf_revenue + ipo_revenue AS total_revenue,
        fno_revenue
    FROM revenue
)

SELECT
    scenario,
    decline_pct,
    ROUND(total_revenue, 2) AS baseline_revenue,
    ROUND(fno_revenue * decline_pct / 100, 2) AS revenue_lost,
    ROUND(
        total_revenue - fno_revenue * decline_pct / 100, 2
    ) AS revised_revenue,
    ROUND(
        100.0 * fno_revenue * decline_pct / 100 /
        total_revenue, 2
    ) AS total_revenue_impact_pct
FROM base
CROSS JOIN (
    SELECT 'Mild' AS scenario, 10 AS decline_pct
    UNION ALL
    SELECT 'Moderate', 25
    UNION ALL
    SELECT 'Severe', 40
) scenarios;

In [0]:
%sql
WITH user_revenue AS (
    SELECT
        third_party_id,
        observable_revenue_90d,
        SUM(observable_revenue_90d) OVER () AS total_revenue
    FROM default.groww_day90_outcomes
),

ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY observable_revenue_90d DESC, third_party_id
        ) AS revenue_rank,
        COUNT(*) OVER () AS total_users
    FROM user_revenue
)

SELECT
    CASE
        WHEN revenue_rank <= CEIL(total_users * 0.01) THEN 'Top 1%'
        WHEN revenue_rank <= CEIL(total_users * 0.05) THEN 'Top 5%'
        WHEN revenue_rank <= CEIL(total_users * 0.10) THEN 'Top 10%'
        WHEN revenue_rank <= CEIL(total_users * 0.20) THEN 'Top 20%'
    END AS user_segment,
    COUNT(*) AS users,
    ROUND(SUM(observable_revenue_90d), 2) AS revenue,
    ROUND(
        100.0 * SUM(observable_revenue_90d) / MAX(total_revenue), 2
    ) AS revenue_share_pct
FROM ranked
WHERE revenue_rank <= CEIL(total_users * 0.20)
GROUP BY 1
ORDER BY MIN(revenue_rank);

In [0]:
%sql
SELECT
    products_successful_90d,
    COUNT(*) AS users,
    ROUND(AVG(observable_revenue_90d), 2) AS revenue_per_user,
    ROUND(SUM(observable_revenue_90d), 2) AS total_revenue,
    ROUND(
        100.0 * SUM(observable_revenue_90d) /
        SUM(SUM(observable_revenue_90d)) OVER (), 2
    ) AS revenue_share_pct
FROM default.groww_day90_outcomes
GROUP BY products_successful_90d
ORDER BY products_successful_90d;

In [0]:
%sql
SELECT
    high_value_90d,
    COUNT(*) AS users,

    ROUND(AVG(stocks_successful_90d), 2) AS avg_stock_orders,
    ROUND(AVG(fno_executed_90d), 2) AS avg_fno_orders,
    ROUND(AVG(mf_completed_90d), 2) AS avg_mf_orders,
    ROUND(AVG(ipo_successful_90d), 2) AS avg_ipo_successful,

    ROUND(AVG(stocks_revenue_90d), 2) AS avg_stock_revenue,
    ROUND(AVG(fno_revenue_90d), 2) AS avg_fno_revenue,
    ROUND(AVG(mf_revenue_90d), 2) AS avg_mf_revenue

FROM default.groww_day90_outcomes
GROUP BY high_value_90d
ORDER BY high_value_90d DESC;